Long-term impact results of PAH toxicity from MPRI2 spill scenarios at Turn Point/Haro Strait

In [ ]:
import os
import xarray as xr
import numpy as np
import itertools
import pandas as pd
import seaborn as sns
from pathlib import Path
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import ssam_groups as groups
from mpl_toolkits.axes_grid1 import make_axes_locatable

### Define scenario and control files

| Scenarios	|	Description |
|-----------|---------------|
| 1	|	no management |
| 2 |   30-day fisheries closures |
| 3	|	90-day fisheries closures |
| 4	|	spill containment within 48hrs |
| 5	|	spill containment within 48hrs + 30-day fisheries closures |
| 6	|	spill containment within 48hrs + 90-day fisheries closures |

8 simulations for each scenario
1. winter S winds
1. winter N winds
1. spring opposing winds & currents
1. spring tandem winds & currents
1. summer Fraser + strong winds
1. summer Fraser + weak winds
1. fall strong N winds
1. fall weak winds

In [ ]:
# Read in salish sea atlantis output files.
control_file = "/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/control-2039-2042/outputSalishSea.nc"
control = xr.open_dataset(str(control_file), decode_cf=True)
time = np.ma.filled(control.variables['t'])


In [ ]:
scenario_root = Path('/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/')
scenario_paths = sorted([p for p in scenario_root.glob('highres-2039-2042_5b*-01-*/outputSalishSea.nc')])
for path in scenario_paths:
    print(path.parent.stem)

highres-2039-2042_5b_030_2019-01-20_closure30
highres-2039-2042_5b_090_2019-01-20_closure90
highres-2039-2042_5b_1_2019-01-20
highres-2039-2042_5b_1_2020-01-24
highres-2039-2042_5b_2_2019-01-20
highres-2039-2042_5b_2_2020-01-24
highres-2039-2042_5b_3_2019-01-20
highres-2039-2042_5b_3_2020-01-24
highres-2039-2042_5b_4_2019-01-20
highres-2039-2042_5b_4_2020-01-24
highres-2039-2042_5b_5_2019-01-20
highres-2039-2042_5b_5_2020-01-24
highres-2039-2042_5b_6_2019-01-20
highres-2039-2042_5b_6_2020-01-24


In [ ]:
scenario_datasets = [xr.open_dataset(scen,decode_cf=True) for scen in scenario_paths]

In [ ]:
# time after burn-in
start = 0
end = time.size-1

### Calculate Mean of Final 3 years (2039-2042)

In [ ]:
def mean_data_pelagic(bio_group, location=groups.salish_sea):
    results = []
    for scenario, path in zip(scenario_datasets,scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]+'_'+groups.scenarios[nm[2]]+'_'+groups.conditions[nm[3]]

        for species in bio_group: 
            p_oiled = np.ma.filled(scenario.variables[bio_group[species] + '_N'][start:end, location, 0:6], np.nan)
            p_control = np.ma.filled(control.variables[bio_group[species] + '_N'][start:end, location, 0:6], np.nan)
            p_oiled = p_oiled.sum(axis=(1,2)).mean()
            p_control = p_control.sum(axis=(1,2)).mean()
            p_ratio = (p_oiled / p_control - 1) * 100
            
            results.append({
            'bio_group': species,
            'scenario': scenario_name,
            'percent_change': p_ratio,
            'sensitivity': groups.sensitivity[bio_group[species]],
            'sensitivity_negative': -groups.sensitivity[bio_group[species]],
            })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/Scen"+scenario_name+"_"+bio_group[species]+".csv")

In [ ]:
def mean_data_benthic(bio_group, location=groups.salish_sea):
    results = []
    for scenario, path in zip(scenario_datasets,scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]+'_'+groups.scenarios[nm[2]]+'_'+groups.conditions[nm[3]]

        for species in bio_group: 
            p_oiled = np.ma.filled(scenario.variables[bio_group[species] + '_N'][start:end, location], np.nan)
            p_control = np.ma.filled(control.variables[bio_group[species] + '_N'][start:end, location], np.nan)
            p_oiled = p_oiled.sum(axis=1).mean()
            p_control = p_control.sum(axis=1).mean()
            p_ratio = (p_oiled / p_control - 1) * 100
            
            results.append({
            'bio_group': species,
            'scenario': scenario_name,
            'percent_change': p_ratio,
            'sensitivity': groups.sensitivity[bio_group[species]],
            'sensitivity_negative': -groups.sensitivity[bio_group[species]],
            })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/Scen"+scenario_name+"_"+bio_group[species]+".csv")


In [ ]:
def mean_data_vertebrates_all_cohorts(bio_group, location=groups.salish_sea):
    results = []

    for scenario, path in zip(scenario_datasets, scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]+'_'+groups.scenarios[nm[2]]+'_'+groups.conditions[nm[3]]

        for species in bio_group:
            numCohorts = groups.cohorts[bio_group[species]]
            sum_ratio = 0

            for cohort in range (1, numCohorts+1):

                new_species = bio_group[species] + str(cohort)
            
                o_numbers_tbl = np.ma.filled(scenario.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
                o_structuralN_tbl = np.ma.filled(scenario.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                o_reservedN_tbl = np.ma.filled(scenario.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                c_numbers_tbl = np.ma.filled(control.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
                c_structuralN_tbl = np.ma.filled(control.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                c_reservedN_tbl = np.ma.filled(control.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                o_weightatage_tbl = (o_structuralN_tbl + o_reservedN_tbl) * o_numbers_tbl 
                o_weightatage = o_weightatage_tbl.sum(axis=(1,2)).mean()

                c_weightatage_tbl = (c_structuralN_tbl + c_reservedN_tbl) * c_numbers_tbl 
                c_weightatage = c_weightatage_tbl.sum(axis=(1,2)).mean()

                ratio = (o_weightatage / c_weightatage - 1) * 100
                #print(ratio)
                sum_ratio = sum_ratio + ratio
            
            species_ratio = sum_ratio/numCohorts
            
            results.append({
            'bio_group': species,
            'scenario': scenario_name,
            'percent_change': species_ratio,
            'sensitivity': groups.sensitivity[bio_group[species]],
            'sensitivity_negative': -groups.sensitivity[bio_group[species]],
            })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/Scen"+scenario_name+"_"+bio_group[species]+"_ALLcohorts.csv")

In [ ]:
def mean_data_vertebrate_cohort3(bio_group, location=groups.salish_sea):
    results = []

    for scenario, path in zip(scenario_datasets, scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]+'_'+groups.scenarios[nm[2]]+'_'+groups.conditions[nm[3]]

        for species in bio_group:

            new_species = bio_group[species] + '3'
        
            o_numbers_tbl = np.ma.filled(scenario.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
            o_structuralN_tbl = np.ma.filled(scenario.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
            o_reservedN_tbl = np.ma.filled(scenario.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

            c_numbers_tbl = np.ma.filled(control.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
            c_structuralN_tbl = np.ma.filled(control.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
            c_reservedN_tbl = np.ma.filled(control.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

            o_weightatage_tbl = (o_structuralN_tbl + o_reservedN_tbl) * o_numbers_tbl 
            o_weightatage = o_weightatage_tbl.sum(axis=(1,2)).mean()

            c_weightatage_tbl = (c_structuralN_tbl + c_reservedN_tbl) * c_numbers_tbl 
            c_weightatage = c_weightatage_tbl.sum(axis=(1,2)).mean()

            ratio = (o_weightatage / c_weightatage - 1) * 100

            results.append({
            'bio_group': species,
            'scenario': scenario_name,
            'percent_change': ratio,
            'sensitivity': groups.sensitivity[bio_group[species]],
            'sensitivity_negative': -groups.sensitivity[bio_group[species]],
            })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/Scen"+scenario_name+"_"+bio_group[species]+"_cohort3.csv")


In [ ]:
def box_plot(df, x_lim=None):
    scenarios = df['scenario'].unique()
    bio_groups = df['bio_group'].unique()

    hatches = ['oo', '///', 'xx', '++']
    alphas = [0.5, 0.75, 0.3, 1]

    n_rows = len(bio_groups)
    if n_rows < 7:
        n_rows = 7
    elif n_rows >10:
        n_rows = 15
    else:
        n_rows = n_rows

    fig, ax = plt.subplots(1, 1, figsize=(4, 6), sharey=True)

    bar_width = 0.9 / len(scenarios)  
    y_pos = np.arange(len(bio_groups))

    for j, scenario in enumerate(scenarios):
        df_plot = df[(df['scenario'] == scenario)]
        df_plot = df_plot.set_index('bio_group').reindex(bio_groups)  
        offset = (j - len(scenarios)/2) * bar_width + bar_width/2

        ax.barh(y_pos + offset, df_plot['percent_change'], height=bar_width, label=scenario, alpha=alphas[j], edgecolor='black', hatch=hatches[j])
    ax.barh(bio_groups, df_plot['sensitivity'], color='grey', alpha=0.3)
    ax.barh(bio_groups, df_plot['sensitivity_negative'], color='grey', alpha=0.3)
    ax.xaxis.grid(True)
    xlabels = ax.get_xticklabels()
    plt.setp(xlabels, fontsize=14)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(bio_groups, fontsize=16) 
    ax.set_xlim(x_lim)
    #ax.set_xlabel("Percent Change", fontsize=16)

    fig.legend(scenarios, bbox_to_anchor=(1.5, 0.9),  fontsize=14) #bbox_to_anchor=(0.8, 1.2) ncol=len(scenarios),
    plt.show()

In [ ]:
mean_data_benthic(groups.benthos)

In [ ]:
mean_data_pelagic(groups.planktonic)

In [ ]:
mean_data_vertebrate_cohort3(groups.salmon)

In [ ]:
mean_data_vertebrate_cohort3(groups.named_fish)

In [ ]:
mean_data_vertebrate_cohort3(groups.other_fish)

In [ ]:
results_root = Path('/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/')
results_paths = sorted([p for p in results_root.glob('Scen*.csv')])
mean_data = []
for file in results_paths:
    df1 = pd.read_csv(file)
    mean_data.append(df1[['bio_group', 'scenario', 'percent_change','sensitivity','sensitivity_negative']])

mean_data_df = pd.concat(mean_data, ignore_index=True)
mean_data_df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/mean_data_fisheries_spring.csv")

In [ ]:
mean_data_vertebrate_cohort3(groups.mammals)

In [ ]:
mean_data_vertebrate_cohort3(groups.sharks)

In [ ]:
mean_data_vertebrate_cohort3(groups.birds)

In [ ]:
results_root = Path('/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/')
results_paths = sorted([p for p in results_root.glob('Scen*.csv')])
mean_data = []
for file in results_paths:
    df1 = pd.read_csv(file)
    mean_data.append(df1[['bio_group', 'scenario', 'percent_change','sensitivity','sensitivity_negative']])

mean_data_df = pd.concat(mean_data, ignore_index=True)
mean_data_df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/mean_data_all_spring.csv")